#  Data Cleaning & Feature Engineering
**Project:** Historical Weather India (2000-2024)


In [1]:
import os
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Pandas version: 3.0.3
Numpy version: 2.5.1


In [2]:
# Construct the relative path from notebooks/ directory to data/processed/
file_path = "../data/processed/weather_clean_v1.csv"

# Load the dataset inspected on Day 3
df = pd.read_csv(file_path)

# Create a working copy to prevent accidental modification of the originally loaded DataFrame
weather = df.copy()

print(f"Dataset successfully loaded from: {file_path}")
print(f"Initial working copy shape: {weather.shape}")

Dataset successfully loaded from: ../data/processed/weather_clean_v1.csv
Initial working copy shape: (91320, 12)


In [3]:
# Check initial data type
print("Initial 'date' column data type:", weather["date"].dtype)

# Convert to datetime if it is an object
if weather["date"].dtype == "object" or weather["date"].dtype == "str":
    weather["date"] = pd.to_datetime(weather["date"])

# Verify transformation
print("Verified 'date' column data type:", weather["date"].dtype)
print("Earliest Date:", weather["date"].min())
print("Latest Date:", weather["date"].max())

Initial 'date' column data type: str
Verified 'date' column data type: datetime64[us]
Earliest Date: 2000-01-01 00:00:00
Latest Date: 2024-12-31 00:00:00


In [4]:
# Check for exact duplicate rows
exact_duplicates = weather.duplicated().sum()
print(f"Exact duplicate rows count: {exact_duplicates}")

# Remove exact duplicates if they exist
if exact_duplicates > 0:
    weather = weather.drop_duplicates()
    print(f"Removed duplicates. New shape: {weather.shape}")

# Verify duplicate status is 0
print(f"Verified exact duplicate rows remaining: {weather.duplicated().sum()}")

# Check unique constraints on combination of city + date
city_date_duplicates = weather.duplicated(subset=["city", "date"]).sum()
print(f"Duplicate records for unique city + date combination: {city_date_duplicates}")

# Inspect if any unexpected multi-records exist
if city_date_duplicates > 0:
    print("\n=== INSPECTING CITY-DATE DUPLICATES ===")
    print(weather[weather.duplicated(subset=["city", "date"], keep=False)].sort_values(["city", "date"]))

Exact duplicate rows count: 0
Verified exact duplicate rows remaining: 0
Duplicate records for unique city + date combination: 0


In [5]:
# Inspect raw unique cities
print("Initial Unique Cities:", weather["city"].unique())
print("Initial Count of Unique Cities:", weather["city"].nunique())

# Clean unnecessary trailing or leading spaces
weather["city"] = weather["city"].str.strip()

# Final sorted inspection
print("Standardized Sorted Cities List:", sorted(weather["city"].unique()))

Initial Unique Cities: <ArrowStringArray>
[    'Delhi',    'Mumbai',   'Kolkata',   'Chennai', 'Bangalore', 'Hyderabad',
 'Ahmedabad',      'Pune',    'Jaipur',   'Lucknow']
Length: 10, dtype: str
Initial Count of Unique Cities: 10
Standardized Sorted Cities List: ['Ahmedabad', 'Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Lucknow', 'Mumbai', 'Pune']


In [6]:
# A. Maximum and minimum temperature cross-validation
invalid_temp_bounds = weather[weather["temperature_2m_max"] < weather["temperature_2m_min"]]
print(f"Rows where max temp is less than min temp: {len(invalid_temp_bounds)}")

# B. Apparent temperature cross-validation
invalid_apparent_bounds = weather[weather["apparent_temperature_max"] < weather["apparent_temperature_min"]]
print(f"Rows where apparent max temp is less than apparent min temp: {len(invalid_apparent_bounds)}")

# C. Negative precipitation check
negative_precipitation = weather[weather["precipitation_sum"] < 0]
print(f"Rows with negative precipitation: {len(negative_precipitation)}")

# D. Negative rainfall check
negative_rainfall = weather[weather["rain_sum"] < 0]
print(f"Rows with negative rainfall: {len(negative_rainfall)}")

# E. Negative wind speed check
negative_wind = weather[weather["wind_speed_10m_max"] < 0]
print(f"Rows with negative wind speed: {len(negative_wind)}")

# F. Negative wind gusts check
negative_gusts = weather[weather["wind_gusts_10m_max"] < 0]
print(f"Rows with negative wind gusts: {len(negative_gusts)}")

# G. Wind direction bounds validation (0 to 360 degrees)
invalid_wind_direction = weather[(weather["wind_direction_10m_dominant"] < 0) | (weather["wind_direction_10m_dominant"] > 360)]
print(f"Rows with invalid wind direction degrees: {len(invalid_wind_direction)}")

Rows where max temp is less than min temp: 0
Rows where apparent max temp is less than apparent min temp: 0
Rows with negative precipitation: 0
Rows with negative rainfall: 0
Rows with negative wind speed: 0
Rows with negative wind gusts: 0
Rows with invalid wind direction degrees: 0


In [7]:
# Create basic time-based component variables
weather["year"] = weather["date"].dt.year
weather["month"] = weather["date"].dt.month
weather["month_name"] = weather["date"].dt.month_name()
weather["day"] = weather["date"].dt.day

# Create calendar tracking sorting column
weather["month_order"] = weather["date"].dt.month

print("Time-based columns added. Previewing components:")
print(weather[["date", "year", "month", "month_name", "day", "month_order"]].head())

Time-based columns added. Previewing components:
        date  year  month month_name  day  month_order
0 2000-01-01  2000      1    January    1            1
1 2000-01-02  2000      1    January    2            1
2 2000-01-03  2000      1    January    3            1
3 2000-01-04  2000      1    January    4            1
4 2000-01-05  2000      1    January    5            1


In [8]:
# Define analytical regional four-season classification mapping function
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

# Apply the seasonal categories element-wise
weather["season"] = weather["month"].apply(get_season)

print("Seasonal breakdown distribution counts:")
print(weather["season"].value_counts())

Seasonal breakdown distribution counts:
season
Monsoon         30500
Summer          23000
Winter          22570
Post-Monsoon    15250
Name: count, dtype: int64


In [9]:
# Compute daily thermal variances
weather["temperature_range"] = weather["temperature_2m_max"] - weather["temperature_2m_min"]

# Estimate daily midpoints
weather["avg_temperature"] = (weather["temperature_2m_max"] + weather["temperature_2m_min"]) / 2

# Apply logical boolean mask for dynamic rainfall status tracking
weather["rainy_day"] = weather["rain_sum"] > 0

print("Derived weather features successfully engineered.")
print(weather[["temperature_2m_max", "temperature_2m_min", "temperature_range", "avg_temperature", "rainy_day"]].head())

Derived weather features successfully engineered.
   temperature_2m_max  temperature_2m_min  temperature_range  avg_temperature  \
0                19.9                 7.4               12.5            13.65   
1                20.0                 5.5               14.5            12.75   
2                20.1                 6.3               13.8            13.20   
3                19.8                 6.4               13.4            13.10   
4                19.4                 5.3               14.1            12.35   

   rainy_day  
0      False  
1      False  
2      False  
3      False  
4      False  


In [10]:
print("=== DATAFRAME STRUCTURAL INFO ===")
print(weather.info())

print("\n=== MISSING VALUE COUNT ===")
print(weather.isnull().sum())

print(f"\nFinal exact duplicates validation count: {weather.duplicated().sum()}")
print(f"Final dataframe dimensional shape layout: {weather.shape}")
print(f"Final structural column checklist: {weather.columns.tolist()}")

print("\n=== FIRST FIVE ROWS VIEW ===")
print(weather.head())

print("\n=== DESCRIPTIVE SUMMARY STATISTICS ===")
print(weather.describe(include='all'))

=== DATAFRAME STRUCTURAL INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 91320 entries, 0 to 91319
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   city                         91320 non-null  str           
 1   date                         91320 non-null  datetime64[us]
 2   temperature_2m_max           91320 non-null  float64       
 3   temperature_2m_min           91320 non-null  float64       
 4   apparent_temperature_max     91320 non-null  float64       
 5   apparent_temperature_min     91320 non-null  float64       
 6   precipitation_sum            91320 non-null  float64       
 7   rain_sum                     91320 non-null  float64       
 8   weather_code                 91320 non-null  int64         
 9   wind_speed_10m_max           91320 non-null  float64       
 10  wind_gusts_10m_max           91320 non-null  float64       
 11  wind_direction_10m

In [11]:
# Establish robust output target definitions
output_file_path = "../data/processed/weather_analysis_ready.csv"

# Export optimized production data without keeping raw range index configurations
weather.to_csv(output_file_path, index=False)
print(f"Success! Optimized production file generated at: {output_file_path}")

Success! Optimized production file generated at: ../data/processed/weather_analysis_ready.csv
